In [2]:
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==============================
# LOAD DATASET (STABLE VERSION)
# ==============================

df = pd.read_csv("data/raw/train.csv")

# ==============================
# FEATURE ENGINEERING
# ==============================

df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["QualityScore"] = df["OverallQual"] * df["OverallCond"]

# ==============================
# FEATURES
# ==============================

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt",
    "TotalSF",
    "HouseAge",
    "QualityScore"
]

target = "SalePrice"

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ==============================
# SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ==============================
# GRID SEARCH
# ==============================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)

# ==============================
# EVALUATION
# ==============================

preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds)
rmse = rmse ** 0.5
r2 = r2_score(y_test, preds)

print("\n📊 Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

Best parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}

📊 Results:
MAE: 18195.08507043931
RMSE: 29298.55592299648
R2: 0.8880875003244812
